# VIA-Auto: Heart Slices Segmentation with PyTorch

An optimized reimplementation of the U-Net training pipeline using PyTorch.

**Improvements over the TensorFlow notebook:**
- Mixed precision training (AMP) for faster GPU utilization
- Proper train/val/test split (the original only had train/test)
- IoU metric computed correctly per-epoch (the original had a broken IoU function)
- Learning rate scheduling with ReduceLROnPlateau
- Reproducible seeding throughout
- Cleaner data pipeline with torch Dataset/DataLoader
- Model checkpointing with best-validation-loss tracking
- No GPU memory leak from stale tensors

## 1. Setup

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import cv2

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Device selection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

# Mixed precision scaler for AMP
scaler = torch.amp.GradScaler("cuda")


## 2. Data Loading

The heart slices dataset is a NumPy array of shape :
- Channels 0-2: RGB image (0-255)
- Channels 3-6: 4 binary mask channels (0-255)

Update the path below to point to your local copy.

In [ ]:
DATA_PATH = "datasets/heartslices_dataset/Data.npy"

# Load data
data = np.load(DATA_PATH).astype(np.float32)
print(f"Data shape: {data.shape}, dtype: {data.dtype}")

# Split into images and masks
images = data[:, :, :, :3]  # RGB, 0-255
masks = data[:, :, :, 3:]   # 4 mask channels, 0-255

# Normalize: images to [-1, 1], masks to [0, 1]
images = (images / 127.5) - 1.0
masks = masks / 255.0

# Check mask values are binary
print(f"Image range: [{images.min():.3f}, {images.max():.3f}]")
print(f"Mask range:  [{masks.min():.3f}, {masks.max():.3f}]")
print(f"Mask unique: {np.unique(masks[:10])}")

In [ ]:
# Three-way split: train (80%) / val (10%) / test (10%)
# First split off test, then split remainder into train/val
X_temp, X_test, y_temp, y_test = train_test_split(
    images, masks, test_size=0.1, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.111, random_state=SEED  # 0.111 of 90% ~= 10% of total
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

# Visualize a sample
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow((X_test[0] + 1) / 2)  # denormalize for display
axes[0].set_title("Input")
for i in range(4):
    axes[i+1].imshow(y_test[0, :, :, i], cmap="gray")
    axes[i+1].set_title(f"Mask {i}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. PyTorch Dataset & DataLoader

In [ ]:
class HeartSlicesDataset(Dataset):
    """Custom dataset for heart slice images and segmentation masks."""

    def __init__(self, images, masks):
        # Convert to torch tensors: NCHW format
        self.images = torch.from_numpy(images).permute(0, 3, 1, 2)  # NHWC -> NCHW
        self.masks = torch.from_numpy(masks).permute(0, 3, 1, 2)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.masks[idx]


BATCH_SIZE = 32

train_ds = HeartSlicesDataset(X_train, y_train)
val_ds = HeartSlicesDataset(X_val, y_val)
test_ds = HeartSlicesDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

# Free memory
del X_train, y_train
print("Training arrays freed from memory")

## 4. U-Net Model

Same architecture as the TF version: 4 downsample blocks (16, 32, 64, 128 filters),
bottleneck (256), 4 upsample blocks, sigmoid output with 4 channels.
Uses LeakyReLU(0.1) and dropout(0.3) as in the original.

In [ ]:
class DoubleConvBlock(nn.Module):
    def __init__(self, in_channels, n_filters):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, n_filters, 3, padding=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(n_filters, n_filters, 3, padding=1),
            nn.LeakyReLU(0.1),
        )

    def forward(self, x):
        return self.double_conv(x)


class DownsampleBlock(nn.Module):
    def __init__(self, in_channels, n_filters):
        super().__init__()
        self.conv = DoubleConvBlock(in_channels, n_filters)
        self.pool = nn.Sequential(
            nn.MaxPool2d(2),
            nn.Dropout(0.3),
        )

    def forward(self, x):
        f = self.conv(x)
        p = self.pool(f)
        return f, p


class UpsampleBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, n_filters):
        super().__init__()
        self.upconv = nn.ConvTranspose2d(in_channels, n_filters, 3, stride=2, padding=1, output_padding=1)
        self.conv = DoubleConvBlock(n_filters + skip_channels, n_filters)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x, skip):
        x = self.upconv(x)
        x = torch.cat([x, skip], dim=1)
        x = self.dropout(x)
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, n_classes=4):
        super().__init__()
        # Downsample
        self.down1 = DownsampleBlock(in_channels, 16)
        self.down2 = DownsampleBlock(16, 32)
        self.down3 = DownsampleBlock(32, 64)
        self.down4 = DownsampleBlock(64, 128)

        # Bottleneck
        self.bottleneck = DoubleConvBlock(128, 256)

        # Upsample
        self.up4 = UpsampleBlock(256, 128, 128)
        self.up3 = UpsampleBlock(128, 64, 64)
        self.up2 = UpsampleBlock(64, 32, 32)
        self.up1 = UpsampleBlock(32, 16, 16)

        # Output
        self.outconv = nn.Conv2d(16, n_classes, 1)

    def forward(self, x):
        f1, p1 = self.down1(x)
        f2, p2 = self.down2(p1)
        f3, p3 = self.down3(p2)
        f4, p4 = self.down4(p3)
        bottleneck = self.bottleneck(p4)
        u4 = self.up4(bottleneck, f4)
        u3 = self.up3(u4, f3)
        u2 = self.up2(u3, f2)
        u1 = self.up1(u2, f1)
        return self.outconv(u1)  # raw logits; BCEWithLogitsLoss applies sigmoid internally


model = UNet(in_channels=3, n_classes=4).to(device)

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f"U-Net parameters: {n_params:,}")
print(model)

## 5. Training Setup

In [ ]:
import torch.optim as optim
from torch.amp import autocast

EPOCHS = 10
LEARNING_RATE = 1e-3

criterion = nn.BCELoss()  # Binary cross-entropy (matches TF original)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5)


def compute_iou(preds, targets, threshold=0.5, eps=1e-7):
    """Compute mean IoU across all mask channels."""
    pred_bin = (torch.sigmoid(preds) > threshold).float()
    intersection = (pred_bin * targets).sum(dim=(2, 3))
    union = (pred_bin + targets).clamp(0, 1).sum(dim=(2, 3))
    iou = (intersection + eps) / (union + eps)
    return iou.mean().item()


# Training history
history = {"train_loss": [], "val_loss": [], "train_iou": [], "val_iou": []}
best_val_loss = float("inf")

print(f"Training for {EPOCHS} epochs with batch size {BATCH_SIZE}")
print(f"Mixed precision (AMP): enabled")

## 6. Training Loop

In [ ]:
for epoch in range(EPOCHS):
    # --- Training ---
    model.train()
    train_loss = 0.0
    train_iou = 0.0
    train_batches = 0

    for images_batch, masks_batch in train_loader:
        images_batch = images_batch.to(device)
        masks_batch = masks_batch.to(device)

        optimizer.zero_grad()

        with autocast("cuda"):
            outputs = model(images_batch)
            loss = criterion(outputs, masks_batch)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        train_iou += compute_iou(outputs.detach(), masks_batch)
        train_batches += 1

    # --- Validation ---
    model.eval()
    val_loss = 0.0
    val_iou = 0.0
    val_batches = 0

    with torch.no_grad():
        for images_batch, masks_batch in val_loader:
            images_batch = images_batch.to(device)
            masks_batch = masks_batch.to(device)

            with autocast("cuda"):
                outputs = model(images_batch)
                loss = criterion(outputs, masks_batch)

            val_loss += loss.item()
            val_iou += compute_iou(outputs, masks_batch)
            val_batches += 1

    # Averages
    train_loss /= train_batches
    val_loss /= max(val_batches, 1)
    train_iou /= train_batches
    val_iou /= max(val_batches, 1)

    # Record history
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_iou"].append(train_iou)
    history["val_iou"].append(val_iou)

    # Learning rate scheduling
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    # Checkpoint best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "model_best.pt")

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} Val Loss: {val_loss:.4f} | "
          f"Train IoU: {train_iou:.4f} Val IoU: {val_iou:.4f} | "
          f"LR: {current_lr:.2e}")

print("
Training complete.")
print(f"Best validation loss: {best_val_loss:.4f}")

## 7. Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history["train_loss"]) + 1)

# Loss
axes[0].plot(epochs_range, history["train_loss"], label="Train")
axes[0].plot(epochs_range, history["val_loss"], label="Validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BCE Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# IoU
axes[1].plot(epochs_range, history["train_iou"], label="Train")
axes[1].plot(epochs_range, history["val_iou"], label="Validation")
axes[1].set_title("IoU")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Mean IoU")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig("curves_pytorch.png", dpi=150)
plt.show()

## 8. Evaluation & Visualization

Load the best model checkpoint and visualize predictions on the test set.

In [ ]:
# Load best model checkpoint
try:
    model.load_state_dict(torch.load("model_best.pt", weights_only=True))
    print("Loaded best model checkpoint")
except FileNotFoundError:
    print("Warning: model_best.pt not found. Using untrained model (run training first).")
model.eval()

In [ ]:
def display_prediction(image, true_masks, pred_masks, idx=0):
    """Display input image, true masks, and predicted masks side by side."""
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))

    # Row 1: Input image (shown 4 times for alignment)
    img = (image[idx].transpose(1, 2, 0) + 1) / 2  # NCHW -> HWC, denormalize
    for i in range(4):
        axes[0, i].imshow(img)
        axes[0, i].set_title("Input" if i == 0 else "")
        axes[0, i].axis("off")

    # Row 2: True masks
    for i in range(4):
        axes[1, i].imshow(true_masks[idx, i], cmap="gray")
        axes[1, i].set_title(f"True Mask {i}")
        axes[1, i].axis("off")

    # Row 3: Predicted masks (thresholded)
    for i in range(4):
        pred_bin = (pred_masks[idx, i] > 0.5).astype(np.uint8)
        axes[2, i].imshow(pred_bin, cmap="gray")
        axes[2, i].set_title(f"Pred Mask {i}")
        axes[2, i].axis("off")

    fig.tight_layout()
    plt.show()


# Show first test sample
display_prediction(X_test, y_test, predicted_masks, idx=0)

In [ ]:
# Threshold a single mask channel (matching original notebook approach)
first_mask = predicted_masks[0, 3]  # Channel 3 in NCHW format

# Binary threshold at 0.95 (same as original)
_, pred_bin = cv2.threshold(
    first_mask.astype(np.float32), 0.95, 1.0, cv2.THRESH_BINARY
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow((X_test[0].transpose(1, 2, 0) + 1) / 2)
axes[0].set_title("Input Image")
axes[1].imshow(y_test[0, :, :, 3], cmap="gray")
axes[1].set_title("True Mask (ch 3)")
axes[2].imshow(pred_bin, cmap="gray")
axes[2].set_title("Predicted Mask (ch 3, threshold=0.95)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Compute test set IoU for each channel
test_ious = []
model.eval()
with torch.no_grad():
    for images_batch, masks_batch in test_loader:
        images_batch = images_batch.to(device)
        masks_batch = masks_batch.to(device)
        with autocast("cuda"):
            outputs = model(images_batch)
        # Per-channel IoU
        for ch in range(4):
            iou = compute_iou(outputs[:, ch:ch+1].detach(), masks_batch[:, ch:ch+1])
            test_ious.append((ch, iou))

# Average per channel
for ch in range(4):
    ch_ious = [v for c, v in test_ious if c == ch]
    print(f"Channel {ch} IoU: {np.mean(ch_ious):.4f}")
print(f"Overall mean IoU: {np.mean([v for _, v in test_ious]):.4f}")

## Summary

| Feature | TF Notebook (original) | PyTorch Notebook (this) |
|---|---|---|
| Framework | TensorFlow/Keras 2.x | PyTorch 2.x |
| Precision | FP32 | Mixed precision (AMP) |
| Data split | Train/Test (90/10) | Train/Val/Test (80/10/10) |
| IoU metric | Broken (never ran correctly) | Working, per-epoch + per-channel |
| LR schedule | None | ReduceLROnPlateau |
| Checkpointing | Save final only | Best validation loss |
| Reproducibility | Not seeded | Fully seeded |
| Model format | SavedModel/H5 |  state_dict |